In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
path_global = '/Users/bdu/Library/CloudStorage/OneDrive-SharedLibraries-epfl.ch/ENAC-IT - Pilot_iAQ_Shared/Data/'

# meta data

In [3]:
df_meta = pd.read_excel(path_global+'Raw/0001_scholair/generic/240412_SCOLAIR_donnees_batiments.xlsx')[['Ref','Campagne_str','Localite_str','Point_mesure','SCOLAIR_Plus','Ventilation','Context','Built_year','Mes_g_d_pose','Mes_g_d_retr']]
df_meta['Ref'] = df_meta['Ref'].astype(str)

df_meta['SCOLAIR_Plus'] = df_meta['SCOLAIR_Plus'].str.replace('SCO','W1')
df_meta['SCOLAIR_Plus'] = df_meta['SCOLAIR_Plus'].str.replace('SP1','W1')
df_meta['SCOLAIR_Plus'] = df_meta['SCOLAIR_Plus'].str.replace('SP2','W2')
df_meta['SCOLAIR_Plus'] = df_meta['SCOLAIR_Plus'].str.replace('SP3','W3')
df_meta['Period identifier'] = df_meta['Campagne_str']+df_meta['SCOLAIR_Plus']

df_meta['Loc_number'] = df_meta['Ref'].apply(lambda x: x[2:4])
df_meta_subset = df_meta[['Ref','Loc_number','Point_mesure','Period identifier']].drop_duplicates().sort_values(by='Loc_number')
df_meta_subset.rename(columns = {'Loc_number':'Building identifier','Point_mesure':'Room identifier'},inplace=True)
df_meta_subset

,Ref,Building identifier,Room identifier,Period identifier
0,11010104,01,EXT,A21W1
197,13010304,01,NHA,E22W1
99,12010304,01,NHA,H22W1
98,12010204,01,NBA,H22W1
97,12010104,01,EXT,H22W1
...,...,...,...,...
96,11240304,24,NHA,A21W1
95,11240204,24,NBA,A21W1
94,11240104,24,EXT,A21W1
290,13240304,24,NHA,E22W1


# CO2 data

## AQM

In [31]:
import glob
co2 = pd.DataFrame() #storing cleaned data
path = path_global+'Raw/0001_scholair/CO2_aqm/' #where data from this sensor is stored
all_files = glob.glob(path + '*.csv')
for file in all_files:
    identifier = file.split('_')[6].split('.')[0] #extract identifier, eg: '../data/raw/COA_cova/data_COA_11050204.csv'
    df = pd.read_csv(file, index_col=0)
    df.columns = ['Timestamp','CO2, ppm','Temperature, °C','Relative_humidity, %','Relative_humidity_STD_21C, %'] #remove identifer from column names
    df = df[['Timestamp','CO2, ppm','Temperature, °C','Relative_humidity, %']] #keep only relevant data 
    df['Timestamp'] = pd.to_datetime(df['Timestamp'],format='mixed') #convert to datetime format
    df = df.set_index('Timestamp').resample('1h').mean().reset_index() #resample to hourly data
    df['Ref'] = identifier #add identifier to match metadata
    co2 = pd.concat([co2, df]) 
    co2 = co2.sort_values(by=['Ref','Timestamp'])

In [32]:
co2 = co2.merge(df_meta_subset, how='left',on='Ref')
co2 = co2.drop('Ref',axis=1)
co2['equipment'] = 'AQM'
co2

,Timestamp,"CO2, ppm","Temperature, °C","Relative_humidity, %",Building identifier,Room identifier,Period identifier,equipment
0,2022-09-08 20:00:00,408.000000,26.200000,40.300000,01,NBA,A21W1,AQM
1,2022-09-08 21:00:00,400.300000,26.186667,40.746667,01,NBA,A21W1,AQM
2,2022-09-08 22:00:00,395.033333,26.093333,41.363333,01,NBA,A21W1,AQM
3,2022-09-08 23:00:00,389.200000,25.996667,41.793333,01,NBA,A21W1,AQM
4,2022-09-09 00:00:00,388.600000,25.920000,42.126667,01,NBA,A21W1,AQM
...,...,...,...,...,...,...,...,...
348368,2023-06-21 08:00:00,854.700000,27.110000,50.256667,24,NHA,A21W1,AQM
348369,2023-06-21 09:00:00,885.466667,27.780000,47.210000,24,NHA,A21W1,AQM
348370,2023-06-21 10:00:00,527.200000,27.836667,47.173333,24,NHA,A21W1,AQM
348371,2023-06-21 11:00:00,669.000000,28.133333,49.573333,24,NHA,A21W1,AQM


In [33]:
co2.to_csv(path_global+'processed/time_series/0001_scholair_co2_atmotube.csv')

## Rotronic

In [26]:
import glob
co2_rot = pd.DataFrame() #storing cleaned data
path = path_global+'Raw/0001_scholair/CO2_rot/' #where data from this sensor is stored
all_files = glob.glob(path + '*.csv')
for file in all_files:
    identifier = file.split('_')[6].split('.')[0] #extract identifier, eg: '../data/raw/COA_cova/data_COA_11050204.csv'
    df = pd.read_csv(file, index_col=0)
    df.columns = ['Timestamp','CO2, ppm','Temperature, °C','Relative_humidity, %','Relative_humidity_STD_21C, %','Occupation'] #remove identifer from column names
    df = df[['Timestamp','CO2, ppm','Temperature, °C','Relative_humidity, %']] #keep only relevant data 
    df['Timestamp'] = pd.to_datetime(df['Timestamp'],format='mixed') #convert to datetime format
    df = df.set_index('Timestamp').resample('1h').mean().reset_index() #resample to hourly data
    df['Ref'] = identifier #add identifier to match metadata
    co2_rot = pd.concat([co2_rot, df]) 
    co2_rot = co2_rot.sort_values(by=['Ref','Timestamp'])

/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/2294529889.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  co2_rot = pd.concat([co2_rot, df])
/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/2294529889.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  co2_rot = pd.concat([co2_rot, df])
/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/2294529889.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is de

In [28]:
co2_rot = co2_rot.merge(df_meta_subset, how='left',on='Ref')
co2_rot = co2_rot.drop('Ref',axis=1)
co2_rot['equipment'] = 'Rotronic'
co2_rot

,Timestamp,"CO2, ppm","Temperature, °C","Relative_humidity, %",Building identifier,Room identifier,Period identifier,equipment
0,2021-12-07 12:00:00,467.950000,25.505000,20.615000,01,NBA,A21W1,Rotronic
1,2021-12-07 13:00:00,614.900000,26.723333,21.220000,01,NBA,A21W1,Rotronic
2,2021-12-07 14:00:00,987.100000,28.953333,21.883333,01,NBA,A21W1,Rotronic
3,2021-12-07 15:00:00,710.300000,26.266667,21.866667,01,NBA,A21W1,Rotronic
4,2021-12-07 16:00:00,470.300000,24.090000,21.126667,01,NBA,A21W1,Rotronic
...,...,...,...,...,...,...,...,...
44669,2023-03-16 11:00:00,757.933333,19.883333,22.750000,24,NHA,H23W1,Rotronic
44670,2023-03-16 12:00:00,617.966667,18.663333,23.050000,24,NHA,H23W1,Rotronic
44671,2023-03-16 13:00:00,684.833333,20.610000,23.603333,24,NHA,H23W1,Rotronic
44672,2023-03-16 14:00:00,1166.633333,25.590000,24.293333,24,NHA,H23W1,Rotronic


In [29]:
co2.to_csv(path_global+'processed/time_series/0001_scholair_co2_rotronic.csv')

# PM data

In [36]:
import glob
pm = pd.DataFrame() #storing cleaned data
path = path_global+'Raw/0001_scholair/PM_atm/' #where data from this sensor is stored
all_files = glob.glob(path + '*.csv')
for file in all_files:
    identifier = file.split('_')[6].split('.')[0] #extract identifier, eg: '../data/raw/COA_cova/data_COA_11050204.csv'
    df = pd.read_csv(file, index_col=0)
    try:
        df.columns = ['Timestamp','TVOC, ppm','AQS','Temperature, °C','Relative_humidity, %','Air_pressure, hPa','PM1, μg/m3','PM2.5, μg/m3','PM10, μg/m3','Latitude','Longitude'] #remove identifer from column names
        df = df[['Timestamp','TVOC, ppm','Temperature, °C','Relative_humidity, %','Air_pressure, hPa','PM1, μg/m3','PM2.5, μg/m3','PM10, μg/m3']] #keep only relevant data 
        df['Timestamp'] = pd.to_datetime(df['Timestamp'],format='mixed') #convert to datetime format
        df = df.set_index('Timestamp').resample('1h').mean().reset_index() #resample to hourly data
        df['Ref'] = identifier #add identifier to match metadata
        pm = pd.concat([pm, df]) 
        pm = pm.sort_values(by=['Ref','Timestamp'])
    except:
        pass

/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/3594474277.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pm = pd.concat([pm, df])
/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/3594474277.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pm = pd.concat([pm, df])
/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/3594474277.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a futur

In [38]:
pm = pm.merge(df_meta_subset, how='left',on='Ref')
pm = pm.drop('Ref',axis=1)
pm['equipment'] = 'Atmotube'
pm

,Timestamp,"TVOC, ppm","Temperature, °C","Relative_humidity, %","Air_pressure, hPa","PM1, μg/m3","PM2.5, μg/m3","PM10, μg/m3",Building identifier,Room identifier,Period identifier,equipment
0,2021-11-17 14:00:00,1.267350,10.533333,50.883333,955.664833,4.508475,6.661017,8.474576,02,EXT,A21W1,Atmotube
1,2021-11-17 15:00:00,0.357883,6.850000,63.616667,956.038500,5.850000,7.883333,9.550000,02,EXT,A21W1,Atmotube
2,2021-11-17 16:00:00,0.323317,6.350000,68.216667,956.367000,6.883333,8.550000,9.900000,02,EXT,A21W1,Atmotube
3,2021-11-17 17:00:00,0.287000,6.066667,68.616667,957.081833,6.450000,8.466667,9.950000,02,EXT,A21W1,Atmotube
4,2021-11-17 18:00:00,0.242650,5.866667,66.016667,957.880000,5.233333,7.133333,8.616667,02,EXT,A21W1,Atmotube
...,...,...,...,...,...,...,...,...,...,...,...,...
18623,2023-03-16 11:00:00,0.126017,22.800000,22.233333,933.115333,1.016667,3.316667,5.550000,24,NBA,H23W1,Atmotube
18624,2023-03-16 12:00:00,0.127333,22.050000,20.816667,933.031667,1.000000,2.250000,3.266667,24,NBA,H23W1,Atmotube
18625,2023-03-16 13:00:00,0.121767,22.166667,21.866667,932.497333,1.000000,1.983333,2.750000,24,NBA,H23W1,Atmotube
18626,2023-03-16 14:00:00,0.147650,23.083333,25.233333,932.224333,1.000000,3.833333,6.183333,24,NBA,H23W1,Atmotube


In [39]:
pm.to_csv(path_global+'processed/time_series/0001_scholair_pm_atmotube.csv')

# UFP data

In [45]:
import glob
ufp = pd.DataFrame() #storing cleaned data
path = path_global+'Raw/0001_scholair/UFP_dmi/' #where data from this sensor is stored
all_files = glob.glob(path + '*.csv')
for file in all_files:
    identifier = file.split('_')[6].split('.')[0] #extract identifier, eg: '../data/raw/COA_cova/data_COA_11050204.csv'
    df = pd.read_csv(file, index_col=0)
    try:
        df.columns = ['Timestamp','UFP, #/cm3','UFP_size, nm','LDSA','filter','Diff'] #remove identifer from column names
        df = df[['Timestamp','UFP, #/cm3','UFP_size, nm']] #keep only relevant data 
        df['Timestamp'] = pd.to_datetime(df['Timestamp'],format='mixed') #convert to datetime format
        df = df.set_index('Timestamp').resample('1h').mean().reset_index() #resample to hourly data
        df['Ref'] = identifier #add identifier to match metadata
        ufp = pd.concat([ufp, df]) 
        ufp = ufp.sort_values(by=['Ref','Timestamp'])
    except:
        pass

/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/4061186182.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ufp = pd.concat([ufp, df])
/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/4061186182.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ufp = pd.concat([ufp, df])
/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/4061186182.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a f

In [46]:
ufp = ufp.merge(df_meta_subset, how='left',on='Ref')
ufp = ufp.drop('Ref',axis=1)
ufp['equipment'] = 'DISCmini'
ufp

,Timestamp,"UFP, #/cm3","UFP_size, nm",Building identifier,Room identifier,Period identifier,equipment
0,2021-11-03 14:00:00,4697.227273,42.813636,06,EXT,A21W1,DISCmini
1,2021-11-03 15:00:00,1995.576271,35.747458,06,EXT,A21W1,DISCmini
2,2021-11-03 16:00:00,2350.593220,33.332203,06,EXT,A21W1,DISCmini
3,2021-11-03 17:00:00,4733.372881,33.055932,06,EXT,A21W1,DISCmini
4,2021-11-03 18:00:00,3402.559322,31.586441,06,EXT,A21W1,DISCmini
...,...,...,...,...,...,...,...
3726,2023-02-15 11:00:00,2256.666667,54.066667,18,EXT,H23W3,DISCmini
3727,2023-02-15 12:00:00,2575.166667,48.766667,18,EXT,H23W3,DISCmini
3728,2023-02-15 13:00:00,2291.166667,51.750000,18,EXT,H23W3,DISCmini
3729,2023-02-15 14:00:00,2115.833333,55.416667,18,EXT,H23W3,DISCmini


In [47]:
ufp.to_csv(path_global+'processed/time_series/0001_scholair_ufp_discmini.csv')

# Radon data

In [48]:
import glob
radon = pd.DataFrame() #storing cleaned data
path = path_global+'Raw/0001_scholair/Radon_rsc/' #where data from this sensor is stored
all_files = glob.glob(path + '*.csv')
for file in all_files:
    identifier = file.split('_')[6].split('.')[0] #extract identifier, eg: '../data/raw/COA_cova/data_COA_11050204.csv'
    df = pd.read_csv(file, index_col=0)
    try:
        df.columns = ['Timestamp','Relative_humidity, %','Temperature, °C','Air_pressure, hPa','Radon, bq/m3','Radon, uncertainty'] #remove identifer from column names
        df = df[['Timestamp','Relative_humidity, %','Temperature, °C','Air_pressure, hPa','Radon, bq/m3']] #keep only relevant data 
        df['Timestamp'] = pd.to_datetime(df['Timestamp'],format='mixed') #convert to datetime format
        df = df.set_index('Timestamp').resample('1h').mean().reset_index() #resample to hourly data
        df['Ref'] = identifier #add identifier to match metadata
        radon = pd.concat([radon, df]) 
        radon = radon.sort_values(by=['Ref','Timestamp'])
    except:
        pass

/var/folders/1_/wr6t48696dsc4ngdd51z5jk40000gq/T/ipykernel_29717/85513372.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  radon = pd.concat([radon, df])


In [49]:
radon = ufp.merge(df_meta_subset, how='left',on='Ref')
radon = ufp.drop('Ref',axis=1)
ufp['equipment'] = 'DISCmini'
ufp

,Timestamp,"Relative_humidity, %","Temperature, °C","Air_pressure, hPa","Radon, bq/m3",Ref
0,2021-11-17 13:00:00,43.76408,18.500000,953.792000,24.461841,11020204
1,2021-11-17 14:00:00,41.08736,20.833333,953.250000,79.365083,11020204
2,2021-11-17 15:00:00,39.28592,22.166667,953.792000,55.555558,11020204
3,2021-11-17 16:00:00,38.92352,22.500000,953.972667,23.809525,11020204
4,2021-11-17 17:00:00,39.10472,22.500000,955.056667,31.746033,11020204
...,...,...,...,...,...,...
166,2022-03-16 11:00:00,41.08928,21.000000,922.898000,42.517006,12180201
167,2022-03-16 12:00:00,40.36976,21.000000,922.356000,34.013605,12180201
168,2022-03-16 13:00:00,40.00928,21.000000,921.814000,42.631917,12180201
169,2022-03-16 14:00:00,40.00928,21.000000,921.091333,51.020407,12180201
